# Gold — Monthly Cost by Team

Agrega custo real vs budget por time, provedor e mês.

**Métricas:**
- `total_cost_usd` — custo acumulado no mês
- `budget_utilization_pct` — (custo/budget) × 100
- `is_over_budget` — flag de estouro de orçamento

**KPIs alimentados:** Total Spend MTD, Budget Utilization %, Teams Over Budget

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
spark = create_spark_session("gold_monthly_cost_by_team")

df_entries = spark.read.format("delta").load("s3a://datalake/silver/cost_entries/")
df_budgets = spark.read.format("delta").load("s3a://datalake/silver/budgets/")

df_cost = (
    df_entries
    .groupBy("team_id", "team_name", "provider", "year", "month")
    .agg(F.round(F.sum("cost_usd"), 4).alias("total_cost_usd"))
)

df_gold = (
    df_cost
    .join(
        df_budgets.select(
            "team_id", "year", "month", "provider",
            F.col("amount_usd").alias("budget_usd")
        ),
        on=["team_id", "year", "month", "provider"],
        how="left",
    )
    .withColumn(
        "budget_utilization_pct",
        F.round((F.col("total_cost_usd") / F.col("budget_usd")) * 100, 2),
    )
    .withColumn("is_over_budget", F.col("total_cost_usd") > F.col("budget_usd"))
    .withColumn("_processed_at", F.current_timestamp())
    .orderBy("year", "month", "team_name", "provider")
)

output_path = "s3a://datalake/gold/monthly_cost_by_team/"
(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(output_path)
)

print(f"[gold_monthly_cost_by_team] {df_gold.count()} linhas → {output_path}")
df_gold.filter(F.col("is_over_budget") == True).select(
    "team_name", "provider", "total_cost_usd", "budget_usd", "budget_utilization_pct"
).orderBy(F.col("budget_utilization_pct").desc()).show()
spark.stop()
